<a href="https://colab.research.google.com/github/ahmedabdelshaheed/flyrank_machinelearning/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedabdelshaheed/flyrank_machinelearning/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*


**Rule.** Rank pages that are old and have a CTR below their peers; then label each page with a reason and an action using simple thresholds.

`score = stale_norm × ctr_gap_norm`

- `stale_norm = percentile_rank(stale_days)`
- `ctr_gap_norm = 1 − percentile_rank(ctr_per_peers)`

**Signals**

- `stale_days`: days from the last update to 2026-03-31.
- `ctr_per_peers`: the page's CTR divided by the average CTR of pages in the same position tier.

**Reason codes** (first match wins)

| Condition | `reason_code` | `action` |
|---|---|---|
| `stale_days` ≤ 30 | `WITHIN_COOLDOWN` | `monitor` |
| `ctr_per_peers` < 0.5 | `CTR_BELOW_POSITION_PEERS` | `snippet_fix` |
| `stale_days` > 180 | `STALE_CONTENT` | `refresh` |
| otherwise | `NO_FLAG` | (empty) |

### Step 1: Load the slice
Load the pages that pass the filters: at least 100 impressions in March, and only clients whose `gsc_data_start` is before 2026-03-01.

In [2]:
import duckdb, pandas as pd
from getpass import getpass
from huggingface_hub import login

TOKEN = getpass("HF token: ")
login(token=TOKEN)

con = duckdb.connect()
con.execute(f"CREATE SECRET hf (TYPE huggingface, TOKEN '{TOKEN}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
SRC_DAILY   = f"{BASE}/fact_content_daily_performance/**/*.parquet"
SRC_CLIENTS = f"{BASE}/dim_clients.parquet"
SRC_CONTENT = f"{BASE}/dim_content.parquet"

START, END  = "2026-01-01", "2026-02-28"
MIN_IMPR = 100   # الحد الأدنى للـ impressions في شهر 3

con.execute(f"""
CREATE OR REPLACE TEMP TABLE content_slice AS
SELECT d.client_hash_id, d.content_hash_id
FROM read_parquet('{SRC_DAILY}', hive_partitioning=true) d
JOIN read_parquet('{SRC_CLIENTS}') c USING (client_hash_id)
WHERE d.month IN ('2026-01', '2026-02')
  AND d.report_date BETWEEN DATE '{START}' AND DATE '{END}'
  AND CAST(c.gsc_data_start AS DATE) < DATE '{START}'
GROUP BY d.client_hash_id, d.content_hash_id
HAVING SUM(d.gsc_impressions) >= {MIN_IMPR}
""")

slice_df = con.execute("SELECT * FROM content_slice").df()

print(len(slice_df))
display(slice_df)

HF token: ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

82612


,client_hash_id,content_hash_id
0,client_e547b89c05043229,content_cc02e6a397d2ffc2
1,client_e547b89c05043229,content_52d68cbbd7862053
2,client_e547b89c05043229,content_782d0c9cd351dcb4
3,client_e547b89c05043229,content_7916e8f527f1c672
4,client_e547b89c05043229,content_e79bcd3d80e36f91
...,...,...
82607,client_b10cb2997d0c7c86,content_5771ae55202a3af9
82608,client_b10cb2997d0c7c86,content_81dbad18ab8abda7
82609,client_b10cb2997d0c7c86,content_d58831c26bee39bc
82610,client_b10cb2997d0c7c86,content_c3a90ecb3b1dcff6


### Step 2: Load the signal tables and the label
Load the two signals, `stale_days` and `ctr_per_peers`, both computed from March only. Then load the label (Down / Up / Stable), computed from April versus March (Impressions).

In [3]:
stale_df = con.execute(f"""
SELECT
    d.content_hash_id AS content,
    s.client_hash_id AS client,
    DATE_DIFF(
        'day',
        CAST(d.content_created_date AS DATE),
        DATE '2026-02-28'
    ) AS stale_days
FROM read_parquet('{SRC_CONTENT}') d
JOIN slice_df s USING (content_hash_id)
WHERE d.content_created_date IS NOT NULL
""").df()



ctr_peers_df = con.execute(f"""
WITH page_data AS (
    SELECT
        s.content_hash_id AS content,
        s.client_hash_id AS client,
        SUM(d.gsc_clicks) * 1.0
            / NULLIF(SUM(d.gsc_impressions), 0) AS ctr,
        AVG(d.gsc_avg_position) AS avg_position
    FROM slice_df s
    LEFT JOIN read_parquet('{SRC_DAILY}', hive_partitioning=true) d
      ON d.content_hash_id = s.content_hash_id
     AND d.client_hash_id = s.client_hash_id
     AND d.month IN ('2026-01', '2026-02')
     AND d.report_date BETWEEN DATE '2026-01-01' AND DATE '2026-02-28'
    GROUP BY s.content_hash_id, s.client_hash_id
),
peer_data AS (
    SELECT
        content,
        client,
        ctr,
        CASE
            WHEN avg_position <= 3 THEN 'top_3'
            WHEN avg_position <= 10 THEN 'page_1'
            WHEN avg_position <= 20 THEN 'striking'
            WHEN avg_position <= 50 THEN 'page_3_5'
            ELSE 'deep'
        END AS position_tier
    FROM page_data
),
peer_average AS (
    SELECT
        content,
        client,
        ctr,
        (
            SUM(ctr) OVER (PARTITION BY position_tier) - ctr
        ) / NULLIF(
            COUNT(*) OVER (PARTITION BY position_tier) - 1, 0
        ) AS peers_avg_ctr
    FROM peer_data
)
SELECT
    content,
    client,
    CASE WHEN peers_avg_ctr > 0
         THEN ctr / peers_avg_ctr
    END AS ctr_per_peers
FROM peer_average

""").df()


label_df = con.execute(f"""
WITH monthly AS (
    SELECT
        d.content_hash_id AS content,
        d.client_hash_id AS client,
        AVG(CASE WHEN d.month IN ('2026-01', '2026-02')
                 THEN d.gsc_impressions END) AS avg_period1,
        COALESCE(
            AVG(CASE WHEN d.month IN ('2026-03', '2026-04')
                     THEN d.gsc_impressions END), 0
        ) AS avg_period2
    FROM read_parquet('{SRC_DAILY}', hive_partitioning=true) d
    JOIN slice_df s USING (content_hash_id, client_hash_id)
    WHERE d.month IN ('2026-01', '2026-02', '2026-03', '2026-04')
    GROUP BY d.content_hash_id, d.client_hash_id
)
SELECT
    content,
    client,
    CASE
        WHEN avg_period1 IS NULL OR avg_period1 = 0
            THEN NULL
        WHEN (avg_period2 - avg_period1) / avg_period1 > 0.20
            THEN 'Up'
        WHEN (avg_period2 - avg_period1) / avg_period1 < -0.20
            THEN 'Down'
        ELSE 'Stable'
    END AS label
FROM monthly
""").df()





display(stale_df)
display(ctr_peers_df)
display(label_df)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content,client,stale_days
0,content_14c17f59aa610ab3,client_2094c6eb080311d5,11
1,content_15770c63daac443b,client_2094c6eb080311d5,24
2,content_157db5a38382c639,client_2094c6eb080311d5,24
3,content_16459ed7545d1aee,client_2094c6eb080311d5,24
4,content_16515c58d44f1f37,client_2094c6eb080311d5,47
...,...,...,...
82607,content_fc7580c9cf7dc181,client_b10cb2997d0c7c86,192
82608,content_fccff79bf5a1fe67,client_b10cb2997d0c7c86,176
82609,content_fce54ccf7952a49f,client_b10cb2997d0c7c86,154
82610,content_ff7283bfa066e6bc,client_b10cb2997d0c7c86,207


,content,client,ctr_per_peers
0,content_a83be3bc17b5eeed,client_73cda7b4e4f265ea,0.000000
1,content_9320dcfc3c4e8083,client_c182d11e4862a37d,0.000000
2,content_6cede8d01e1755e0,client_62f4a7e64f5e0096,0.000000
3,content_66186933deb20d70,client_3197e6291363b4db,0.000000
4,content_fab5cf92f93a6839,client_e547b89c05043229,0.000000
...,...,...,...
82607,content_9fa132b5d23fca8f,client_2094c6eb080311d5,1.225027
82608,content_ec0badd96b07078e,client_2094c6eb080311d5,0.000000
82609,content_8fb3c020b0b5abd0,client_e547b89c05043229,0.861189
82610,content_3a9accde032678ad,client_e547b89c05043229,1.037733


,content,client,label
0,content_52d68cbbd7862053,client_e547b89c05043229,Up
1,content_782d0c9cd351dcb4,client_e547b89c05043229,Up
2,content_c7c2293b1bf9d32c,client_e547b89c05043229,Up
3,content_2741b9688e995027,client_e547b89c05043229,Up
4,content_ac750bab14ef9b9b,client_e547b89c05043229,Stable
...,...,...,...
82607,content_105c5ccf15503807,client_2094c6eb080311d5,Up
82608,content_ad025cec5f892be8,client_2094c6eb080311d5,Down
82609,content_cbcce52051a18b10,client_b10cb2997d0c7c86,Down
82610,content_81dbad18ab8abda7,client_b10cb2997d0c7c86,Up


### Step 3: Test the signals
One small bucket table per signal, with the number of pages (n) and the share of Down / Up / Stable in each bucket. Each signal gets a one-word verdict.

In [4]:
# Stale summary
stale_analysis = label_df.merge(stale_df, on=["content", "client"])

stale_analysis["stale_range"] = pd.cut(
    stale_analysis["stale_days"],
    bins=[-float("inf"), 30, 90, 180, float("inf")],
    labels=["0-30", "31-90", "91-180", "180+"]
)

stale_counts = stale_analysis.groupby(
    ["stale_range", "label"], observed=False
).size().unstack(fill_value=0)

stale_counts = stale_counts.reindex(
    columns=["Down", "Up", "Stable"], fill_value=0
)

stale_summary = stale_counts.div(
    stale_counts.sum(axis=1).replace(0, float("nan")), axis=0
).mul(100).round(1)

stale_summary["Count"] = stale_counts.sum(axis=1)
stale_summary = stale_summary.reset_index()

display(stale_summary)


# CTR per peers summary
peers_analysis = label_df.merge(
    ctr_peers_df, on=["content", "client"]
)

peers_analysis["ctr_peers_range"] = pd.cut(
    peers_analysis["ctr_per_peers"],
    bins=[-float("inf"), 0.5, 0.8, 1.2, float("inf")],
    labels=["Very Low", "Low", "Average", "High"]
)

peers_counts = peers_analysis.groupby(
    ["ctr_peers_range", "label"], observed=False
).size().unstack(fill_value=0)

peers_counts = peers_counts.reindex(
    columns=["Down", "Up", "Stable"], fill_value=0
)

peers_summary = peers_counts.div(
    peers_counts.sum(axis=1).replace(0, float("nan")), axis=0
).mul(100).round(1)

peers_summary["Count"] = peers_counts.sum(axis=1)
peers_summary = peers_summary.reset_index()

display(peers_summary)

label,stale_range,Down,Up,Stable,Count
0,0-30,26.0,58.1,15.9,3739
1,31-90,29.9,53.1,17.0,12003
2,91-180,37.6,40.0,22.4,18050
3,180+,33.3,40.2,26.6,48820


label,ctr_peers_range,Down,Up,Stable,Count
0,Very Low,39.7,38.5,21.8,40734
1,Low,28.4,46.1,25.5,10500
2,Average,28.6,43.7,27.7,9423
3,High,26.0,49.0,25.0,21955


- **CTR vs peers:** CONFIRMED.
- **Staleness:** MIXED.

### Step 4: Build the final table
Normalize both signals, compute the score, then assign the reason code and the action. The table is sorted by score.

In [5]:
import numpy as np

COOLDOWN_DAYS = 30
STALE_DAYS    = 180
CTR_RATIO     = 0.5

rule_df = stale_df.merge(ctr_peers_df, on=["content", "client"])


rule_df["age_norm"]     = rule_df["stale_days"].rank(pct=True)
rule_df["ctr_gap_norm"] = 1 - rule_df["ctr_per_peers"].rank(pct=True)

# score
rule_df["score"] = rule_df["age_norm"] * rule_df["ctr_gap_norm"]

# reason code + action
cooldown = rule_df["stale_days"] <= COOLDOWN_DAYS
low_ctr  = rule_df["ctr_per_peers"] < CTR_RATIO
stale    = rule_df["stale_days"] > STALE_DAYS

conds   = [cooldown, low_ctr & stale, low_ctr, stale]
reasons = ["WITHIN_COOLDOWN", "CTR_BELOW_POSITION_PEERS + STALE_CONTENT", "CTR_BELOW_POSITION_PEERS", "STALE_CONTENT"]
actions = ["monitor", "snippet_fix + refresh", "snippet_fix", "refresh"]

rule_df["reason_code"] = np.select(conds, reasons, default="")
rule_df["action"]      = np.select(conds, actions, default="")

rule_df = rule_df.sort_values("score", ascending=False)
rule_df[["content", "client", "stale_days", "ctr_per_peers", "score", "reason_code", "action"]].head(10)

,content,client,stale_days,ctr_per_peers,score,reason_code,action
67483,content_a1a50ac7d5ad77ad,client_c182d11e4862a37d,463,0.0,0.829386,CTR_BELOW_POSITION_PEERS + STALE_CONTENT,snippet_fix + refresh
67133,content_12510268ee8809ee,client_c182d11e4862a37d,456,0.0,0.827504,CTR_BELOW_POSITION_PEERS + STALE_CONTENT,snippet_fix + refresh
67414,content_825b41eb7b30bfc3,client_c182d11e4862a37d,456,0.0,0.827504,CTR_BELOW_POSITION_PEERS + STALE_CONTENT,snippet_fix + refresh
67413,content_8210228e0371fbb9,client_c182d11e4862a37d,456,0.0,0.827504,CTR_BELOW_POSITION_PEERS + STALE_CONTENT,snippet_fix + refresh
67410,content_812e8ccc6d1dc942,client_c182d11e4862a37d,456,0.0,0.827504,CTR_BELOW_POSITION_PEERS + STALE_CONTENT,snippet_fix + refresh
67409,content_80fb61a23116d5e4,client_c182d11e4862a37d,456,0.0,0.827504,CTR_BELOW_POSITION_PEERS + STALE_CONTENT,snippet_fix + refresh
67443,content_920d5d8614441cec,client_c182d11e4862a37d,456,0.0,0.827504,CTR_BELOW_POSITION_PEERS + STALE_CONTENT,snippet_fix + refresh
67401,content_7e58eb5e3aa4a9bb,client_c182d11e4862a37d,456,0.0,0.827504,CTR_BELOW_POSITION_PEERS + STALE_CONTENT,snippet_fix + refresh
67394,content_7be7b3c0eec34496,client_c182d11e4862a37d,456,0.0,0.827504,CTR_BELOW_POSITION_PEERS + STALE_CONTENT,snippet_fix + refresh
67392,content_7a7fa39e10ceb205,client_c182d11e4862a37d,456,0.0,0.827504,CTR_BELOW_POSITION_PEERS + STALE_CONTENT,snippet_fix + refresh


In [7]:
KS = [100, 200, 500, 1000]

eval_df = rule_df.merge(label_df, on=["content", "client"])
eval_df = eval_df.sort_values("score", ascending=False)

base_rate = (eval_df["label"] == "Down").mean()
print(f"base rate (Down): {base_rate:.1%}")

for K in KS:
    top_k = eval_df.head(K)
    precision_at_k = (top_k["label"] == "Down").mean()
    print(f"precision@{K}: {precision_at_k:.1%}")



base rate (Down): 33.4%
precision@100: 86.0%
precision@200: 82.5%
precision@500: 54.0%
precision@1000: 48.3%


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
import os, base64, requests

PUSH_TO_GITHUB = False

PATH = "work/outputs/baseline_action_score.csv"
q = rule_df[["content", "client", "stale_days", "ctr_per_peers", "score", "reason_code", "action"]].reset_index(drop=True)
q.index += 1

# Colab
os.makedirs("work/outputs", exist_ok=True)
q.to_csv(PATH, index_label="rank")

# GitHub
if PUSH_TO_GITHUB:
    from google.colab import userdata
    URL = f"https://api.github.com/repos/ahmedabdelshaheed/flyrank_machinelearning/contents/{PATH}"
    H = {"Authorization": f"Bearer {userdata.get('GITHUB_TOKEN')}", "Accept": "application/vnd.github+json"}
    body = {"message": "Add baseline_action_score.csv", "content": base64.b64encode(open(PATH, "rb").read()).decode()}
    old = requests.get(URL, headers=H)
    if old.status_code == 200:
        body["sha"] = old.json()["sha"]
    print(requests.put(URL, headers=H, json=body).status_code)

201


In [15]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [16]:
rule_df.to_csv('/content/drive/MyDrive/my_dataframe.csv', index=False)

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [13]:
rule_df[["content", "client", "stale_days", "ctr_per_peers", "score", "reason_code", "action"]].head(20)

,content,client,stale_days,ctr_per_peers,score,reason_code,action
67483,content_a1a50ac7d5ad77ad,client_c182d11e4862a37d,463,0.0,0.829386,CTR_BELOW_POSITION_PEERS + STALE_CONTENT,snippet_fix + refresh
67133,content_12510268ee8809ee,client_c182d11e4862a37d,456,0.0,0.827504,CTR_BELOW_POSITION_PEERS + STALE_CONTENT,snippet_fix + refresh
67414,content_825b41eb7b30bfc3,client_c182d11e4862a37d,456,0.0,0.827504,CTR_BELOW_POSITION_PEERS + STALE_CONTENT,snippet_fix + refresh
67413,content_8210228e0371fbb9,client_c182d11e4862a37d,456,0.0,0.827504,CTR_BELOW_POSITION_PEERS + STALE_CONTENT,snippet_fix + refresh
67410,content_812e8ccc6d1dc942,client_c182d11e4862a37d,456,0.0,0.827504,CTR_BELOW_POSITION_PEERS + STALE_CONTENT,snippet_fix + refresh
67409,content_80fb61a23116d5e4,client_c182d11e4862a37d,456,0.0,0.827504,CTR_BELOW_POSITION_PEERS + STALE_CONTENT,snippet_fix + refresh
67443,content_920d5d8614441cec,client_c182d11e4862a37d,456,0.0,0.827504,CTR_BELOW_POSITION_PEERS + STALE_CONTENT,snippet_fix + refresh
67401,content_7e58eb5e3aa4a9bb,client_c182d11e4862a37d,456,0.0,0.827504,CTR_BELOW_POSITION_PEERS + STALE_CONTENT,snippet_fix + refresh
67394,content_7be7b3c0eec34496,client_c182d11e4862a37d,456,0.0,0.827504,CTR_BELOW_POSITION_PEERS + STALE_CONTENT,snippet_fix + refresh
67392,content_7a7fa39e10ceb205,client_c182d11e4862a37d,456,0.0,0.827504,CTR_BELOW_POSITION_PEERS + STALE_CONTENT,snippet_fix + refresh



All 20 rows are reviewed together as one group.

- **Reason codes:** `CTR_BELOW_POSITION_PEERS` and `STALE_CONTENT`.
- **Actions:** `snippet_fix` and `refresh`.
- **Staleness is not proof of a refresh:** an old page can still be accurate, so a person should open the page before refreshing it.
- **One-month window:** the signals come from 30 days only, so a seasonal dip or random noise could explain some of them.
- **Single client:** all 20 pages belong to one client (and the same client holds the top 200 ranks). This may be a client-level issue, such as a shared title template, but I could not verify it.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

all features end on 2026-03-31. April is used only to check the signals or to evaluate (precision@k, ROC AUC), not to build the score.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.